In [2]:
# in a Jupyter cell
import re, json, asyncio, time
from playwright.async_api import async_playwright

SD_SEARCH_URL = "https://www.sciencedirect.com/search?qs=fatty+liver&years=2026%2C2025&lastSelectedFacet=accessTypes&articleTypes=FLA&langs=en&subjectAreas=2700%2C3000&accessTypes=openaccess"

async def extract_articles(url, max_items=50):
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, timeout=60000)
        await page.wait_for_selector("ol#search-results-list, .ResultList", timeout=20000)
        for _ in range(3):
            await page.mouse.wheel(0, 10000)
            await asyncio.sleep(1.0)
        nodes = await page.query_selector_all("ol#search-results-list li, .ResultItem")
        for n in nodes[:max_items]:
            title_el = await n.query_selector("h2, h3, .result-item-title, .title")
            link_el = await n.query_selector("a, a.title")
            title = await title_el.inner_text() if title_el else None
            url_ = None
            doi = None
            if link_el:
                href = await link_el.get_attribute("href")
                if href and href.startswith("/"):
                    url_ = "https://www.sciencedirect.com" + href
                else:
                    url_ = href
                if url_:
                    m = re.search(r"10\.\d{4,9}/[-._;()/:A-Z0-9]+", url_, re.I)
                    if m:
                        doi = m.group(0)
            doi_el = await n.query_selector("a[href*='doi'], span.doi, .doi")
            if doi_el and not doi:
                txt = await doi_el.inner_text()
                m = re.search(r"10\.\d{4,9}/[-._;()/:A-Z0-9]+", txt or "", re.I)
                if m:
                    doi = m.group(0)
            results.append({"title": title.strip() if title else None, "article_url": url_, "doi": doi})
        await browser.close()
    return results

# then in the same cell or next cell:
items = await extract_articles(SD_SEARCH_URL, max_items=100)
print(json.dumps(items, indent=2))


NotImplementedError: 

In [ ]:
# save as unpaywall_download.py
import requests
import time
import os

UNPAYWALL = "https://api.unpaywall.org/v2"
EMAIL = "tejas.sanju@inverv.com"   # Unpaywall requires an email param for polite usage

def get_oa_location(doi):
    url = f"{UNPAYWALL}/{doi}"
    params = {"email": EMAIL}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code == 200:
        return r.json()
    else:
        print("Unpaywall error", r.status_code, r.text)
        return None

def download_best_pdf(doi, out_dir="pdfs"):
    data = get_oa_location(doi)
    if not data:
        return None
    # Unpaywall returns 'best_oa_location' and list 'oa_locations'
    loc = data.get("best_oa_location") or (data.get("oa_locations") or [None])[0]
    if not loc:
        print("No OA location found for", doi)
        return None
    pdf_url = loc.get("url_for_pdf") or loc.get("url")
    if not pdf_url:
        print("No PDF URL available in OA location for", doi)
        return None

    os.makedirs(out_dir, exist_ok=True)
    local_fname = os.path.join(out_dir, doi.replace("/", "_") + ".pdf")
    print("Downloading:", pdf_url)
    with requests.get(pdf_url, stream=True, timeout=30) as r:
        if r.status_code == 200 and 'application/pdf' in r.headers.get('Content-Type',''):
            with open(local_fname, "wb") as f:
                for chunk in r.iter_content(1024*64):
                    f.write(chunk)
            print("Saved:", local_fname)
            # polite pause
            time.sleep(1.0)
            return local_fname
        else:
            print("Failed to fetch PDF or not a PDF (status, content-type):", r.status_code, r.headers.get('Content-Type'))
            return None

# example:
if __name__ == "__main__":
    doi = "10.1016/j.xphs.2025.01.001"  # replace with real DOI
    download_best_pdf(doi)
